In [ ]:
!apt-get install swig
!pip install gymnasium[box2d]
!pip install gym
!pip install pygame

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 49 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 1s (1,409 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 123632 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubu

In [ ]:
import os
import warnings
import logging
from contextlib import redirect_stdout

# Suppress TensorFlow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

# Suppress gym logs
logging.getLogger('gym').setLevel(logging.CRITICAL)

import numpy as np
import random
from collections import deque
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size

        # Hyperparameters
        self.LEARNING_RATE = 0.001
        self.DISCOUNT = 0.95
        self.EPSILON = 1.0
        self.EPSILON_DECAY = 0.995
        self.EPSILON_MIN = 0.01
        self.MEMORY_SIZE = 100000
        self.MIN_REPLAY_MEMORY_SIZE = 1000
        self.MINIBATCH_SIZE = 64
        self.replay_memory = deque(maxlen=self.MEMORY_SIZE)
        self.model = self.build_dqn()

    def build_dqn(self):
        model = Sequential()
        model.add(Dense(128, input_dim=self.state_size, activation='relu'))
        model.add(Dense(128, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(0.2))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.LEARNING_RATE))
        return model

    def update_replay_memory(self, transition):
        self.replay_memory.append(transition)

    def act(self, state):
        q_values = self.model.predict(state[np.newaxis], verbose=0)  # Ensure no output
        if np.random.rand() < self.EPSILON:
            return random.choice(range(self.action_size))
        return np.argmax(q_values[0])

    def train(self):
        if len(self.replay_memory) < self.MIN_REPLAY_MEMORY_SIZE:
            return

        minibatch = random.sample(self.replay_memory, self.MINIBATCH_SIZE)

        current_states = np.array([transition[0] for transition in minibatch])
        actions = np.array([transition[1] for transition in minibatch])
        rewards = np.array([transition[2] for transition in minibatch])
        next_states = np.array([transition[3] for transition in minibatch])
        dones = np.array([transition[4] for transition in minibatch])

        current_qs = self.model.predict(current_states, verbose=0)
        future_qs = self.model.predict(next_states, verbose=0)

        for i in range(self.MINIBATCH_SIZE):
            if not dones[i]:
                current_qs[i][actions[i]] = rewards[i] + self.DISCOUNT * np.max(future_qs[i])
            else:
                current_qs[i][actions[i]] = rewards[i]

        self.model.fit(current_states, current_qs, batch_size=self.MINIBATCH_SIZE, verbose=0)

        if self.EPSILON > self.EPSILON_MIN:
            self.EPSILON *= self.EPSILON_DECAY

import gym

env = gym.make('LunarLander-v2')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = DQNAgent(state_size, action_size)

EPISODES = 50000
SHOW_EVERY = 3000

with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull):  # Suppress all unwanted outputs
        for episode in range(1, EPISODES + 1):
            print('episode: ',episode)
            current_state = env.reset()
            done = False
            total_reward = 0

            while not done:
                action = agent.act(current_state)
                new_state, reward, done, info = env.step(action)
                agent.update_replay_memory((current_state, action, reward, new_state, done))
                agent.train()
                current_state = new_state
                total_reward += reward

            if episode % SHOW_EVERY == 0:
                print(f"Episode: {episode}, Reward: {total_reward}, Epsilon: {agent.EPSILON:.4f}")


/usr/local/lib/python3.10/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa